# Gladiators - Road Damage Detection (HTH-CV-07)
Runtime: **Runtime > Change runtime type > T4 GPU**. Run the cells in order.

Outputs `detections.csv` for the optimizer, dashboard and website.

## 1. Install and download the RDD2022 dataset (Roboflow)
Paste your own Roboflow API key (Roboflow > Settings > API Keys).

In [ ]:
!pip install -q ultralytics roboflow huggingface_hub

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("tdt4265-vd97u").project("rdd2022-nldlk")
dataset = project.version(14).download("yolov8")

!cat {dataset.location}/data.yaml
!echo "Train images:" && ls {dataset.location}/train/images | wc -l
!echo "Valid images:" && ls {dataset.location}/valid/images | wc -l

## 2. (Optional) Train our own YOLOv8s
On free Colab this reached mAP50 = 0.138 after 18 epochs (~6.5 min/epoch). Skip to step 3 if short on time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
own = YOLO('yolov8s.pt')
own.train(data=f"{dataset.location}/data.yaml", epochs=30, imgsz=640, batch=16, patience=10,
          project='/content/drive/MyDrive/road_damage', name='yolov8s_rdd', save_period=5)

## 3. Pretrained RDD2022 YOLOv8s (Hugging Face: dronefreak/rdd2022-yolov8s)
Validate on our split at 640 px and 1024 px.

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

w = hf_hub_download("dronefreak/rdd2022-yolov8s", "best.pt")
model = YOLO(w)
print("CLASS NAMES:", model.names)

m640 = model.val(data=f"{dataset.location}/data.yaml", split="val", plots=False)
m1024 = model.val(data=f"{dataset.location}/data.yaml", split="val", imgsz=1024, plots=False)
print("mAP50 @640 :", round(m640.box.map50, 3))
print("mAP50 @1024:", round(m1024.box.map50, 3))

## 4. Detect damage on 30 road images -> detections.csv
One image = one road segment. Road class and location are simulated for the demo.

In [ ]:
import glob, os, random, numpy as np, pandas as pd
random.seed(0); rng = np.random.default_rng(0)

def to_idx(name):
    n = name.lower()
    if 'd00' in n or 'long' in n:  return 0
    if 'd10' in n or 'trans' in n: return 1
    if 'd20' in n or 'allig' in n: return 2
    return 3

CENTER = (9.9312, 76.2673)
ROADS, P = ['Highway', 'Arterial', 'Collector', 'Local'], [0.15, 0.3, 0.3, 0.25]
imgs = sorted(glob.glob(f"{dataset.location}/valid/images/*")); random.shuffle(imgs)
os.makedirs('annotated', exist_ok=True)

rows, seg = [], 0
for path in imgs:
    r = model.predict(path, conf=0.25, imgsz=1024, verbose=False)[0]
    if len(r.boxes) == 0: continue
    seg += 1; sid = f"SEG-{seg:02d}"
    road = rng.choice(ROADS, p=P)
    lat, lon = CENTER[0] + rng.uniform(-0.04, 0.04), CENTER[1] + rng.uniform(-0.04, 0.04)
    r.save(filename=f"annotated/{sid}.jpg")
    for c, cf, (x, y, w, h) in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xywhn.tolist()):
        rows.append(dict(segment=sid, image=os.path.basename(path), road_type=road, lat=lat, lon=lon,
                         cls=to_idx(model.names[int(c)]), conf=round(cf, 3), area=round(w*h, 4)))
    if seg == 30: break

det = pd.DataFrame(rows); det.to_csv('detections.csv', index=False)
print(det.segment.nunique(), "segments,", len(det), "detections")
print(det.groupby('cls').size())

## 5. View detections

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob('annotated/*.jpg'))[:4]:
    display(Image(f, width=600))

## 6. Download results

In [ ]:
!zip -qr annotated.zip annotated
from google.colab import files
files.download('detections.csv')
files.download('annotated.zip')